In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../../datasets/StudentsPerformance.csv")
df

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


In [3]:
df.replace("?", np.nan, inplace=True)
df.isnull().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [4]:
df.isna().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [5]:
df["parental level of education"].unique()

<StringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str

In [6]:
education_order = {
    "some high school": 0,
    "high school": 1,
    "some college": 2,
    "associate's degree": 3,
    "bachelor's degree": 4,
    "master's degree": 5
}

df["parental level of education"] = df["parental level of education"].map(education_order)

In [7]:
le = LabelEncoder()
df = pd.get_dummies(df, columns=["race/ethnicity", "gender", "lunch", 'test preparation course'], drop_first=True)
df

,parental level of education,math score,reading score,writing score,race/ethnicity_group B,race/ethnicity_group C,race/ethnicity_group D,race/ethnicity_group E,gender_male,lunch_standard,test preparation course_none
0,4,72,72,74,True,False,False,False,False,True,True
1,2,69,90,88,False,True,False,False,False,True,False
2,5,90,95,93,True,False,False,False,False,True,True
3,3,47,57,44,False,False,False,False,True,False,True
4,2,76,78,75,False,True,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
995,5,88,99,95,False,False,False,True,False,True,False
996,1,62,55,55,False,True,False,False,True,False,True
997,1,59,71,65,False,True,False,False,False,False,False
998,2,68,78,77,False,False,True,False,False,True,False


In [10]:
X = df.drop(columns=["math score", "reading score", "writing score"])
Y = df[["math score", "reading score", "writing score"]]

In [11]:
comp = {
    "Degree": [],
    "CV RMSE": []
}
for degree in range(1, 6):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree)),
        ("lr", LinearRegression())
    ])
    scores = cross_val_score(
        model,
        X,
        Y,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )
    comp["Degree"].append(degree)
    comp["CV RMSE"].append((-scores).mean())
comp_df = pd.DataFrame(comp)
comp_df

,Degree,CV RMSE
0,1,12.935062
1,2,13.288348
2,3,13.838505
3,4,14.415815
4,5,15.668954
